# <center style="font-weight: bold; color: #0098cd;">Transcripción automática del habla (ASR)</center>

## 1. Introducción

La conversión de señales de audio en representaciones textuales constituye una etapa crítica en sistemas basados en reconocimiento automático del habla (*Automatic Speech Recognition*, ASR). En entornos reales, incluso tras aplicar técnicas de preprocesamiento, persisten factores como la variabilidad en la pronunciación, el ruido residual o las condiciones de grabación, que pueden afectar directamente a la calidad de las transcripciones generadas.

Este *notebook* aborda la fase de transcripción automática a partir de los audios previamente procesados, así como la evaluación cuantitativa de los resultados obtenidos. Se adopta un enfoque experimental, en el que se comparan diferentes configuraciones de entrada y parámetros del modelo ASR, con el objetivo de analizar su impacto en métricas estándar como *Word Error Rate* (WER) y *Character Error Rate* (CER).

Además, se incorporan mecanismos de análisis que permiten caracterizar no solo el error medio, sino también su distribución, identificando comportamientos extremos y evaluando la robustez del sistema frente a distintos tipos de entrada.

### 1.1 Objetivo

El objetivo de este *notebook* es aplicar un modelo ASR para generar transcripciones automáticas a partir de audios preprocesados, y evaluar de forma sistemática la calidad de dichas transcripciones mediante métricas cuantitativas y análisis comparativo entre distintas configuraciones.

### 1.2 Contexto dentro del sistema completo

La transcripción automática constituye la segunda etapa del sistema, actuando como puente entre el procesamiento de señal y las tareas de análisis semántico posteriores. Su función es transformar la información acústica en texto, que servirá como entrada para los módulos de procesamiento del lenguaje natural (*Natural Language Processing*, NLP), incluyendo clasificación de mensajes y reconocimiento de entidades.

La calidad de esta etapa condiciona directamente el rendimiento del sistema completo, ya que los errores generados en la transcripción se propagan a las fases posteriores. Por este motivo, no solo se evalúa la precisión del modelo ASR, sino también el impacto de dichos errores en términos estructurales y semánticos.

En este sentido, este *notebook* no se limita a medir el rendimiento del modelo ASR de forma aislada, sino que establece las bases para analizar cómo los errores de transcripción afectan a tareas posteriores como la identificación de entidades o la clasificación de eventos. Esta conexión es clave para comprender el comportamiento del sistema en un escenario real.

### 1.3 Requisitos de esta fase

El desarrollo de la fase de transcripción y evaluación ASR se rige por una serie de requisitos técnicos que garantizan la validez y reproducibilidad del análisis.

En primer lugar, es necesario aplicar el modelo ASR de forma consistente sobre diferentes variantes de los audios de entrada, permitiendo comparar el efecto del preprocesamiento previo sobre la calidad de las transcripciones generadas.

En segundo lugar, se requiere implementar un sistema de evaluación cuantitativa basado en métricas estándar como WER y CER, asegurando que estas se calculan sobre representaciones homogéneas mediante una normalización previa del texto que elimine variabilidad superficial sin afectar al contenido semántico.

Asimismo, es fundamental mantener la trazabilidad de los resultados, preservando las transcripciones generadas en su forma original (*raw output*) y almacenándolas de forma estructurada para su posterior análisis y reutilización.

Adicionalmente, el proceso debe permitir el análisis detallado del comportamiento del sistema, incorporando no solo estadísticas agregadas, sino también el estudio de la distribución del error, la identificación de casos extremos y la evaluación del impacto de errores específicos relevantes para el dominio.

Finalmente, se establece como principio de diseño separar claramente las etapas de generación de texto, evaluación y procesamiento lingüístico. Esta decisión permite evitar dependencias implícitas entre módulos y facilita el análisis de la propagación del error a lo largo del *pipeline*, aspecto fundamental para interpretar correctamente los resultados en las etapas de NLP desarrolladas en el siguiente *notebook*.

## 2. Preparación del entorno de trabajo

En esta sección se define el entorno técnico necesario para el desarrollo del pipeline de preprocesamiento de audio. Se incluyen la instalación de dependencias, la importación de librerías especializadas y la configuración de rutas de trabajo.

El objetivo es garantizar un entorno reproducible y estructurado que permita ejecutar el flujo completo de procesamiento sobre diferentes conjuntos de datos sin modificaciones en la lógica del código. Asimismo, se establece una organización clara de los directorios que facilita la separación entre datos originales, datos intermedios y resultados finales.

### 2.1 Instalación de dependencias

En este apartado se especifica la instalación de las librerías necesarias para la ejecución del *notebook*. Todas las dependencias se gestionan mediante un archivo `requirements.txt`, lo que permite replicar el entorno de ejecución de forma controlada y consistente.

#### 2.1.1 Configuración del entorno

Este notebook requiere la instalación previa de las dependencias del proyecto.

Ejecutar en terminal:

```bash
pip install -r requirements.txt

### 2.2 Importación de librerías

In [10]:
# ==============================
# Gestión de rutas y sistema
# ==============================
from pathlib import Path                    # gestión de rutas
import os                                   # operaciones del sistema
import json                                 # lectura/escritura JSON

# ==============================
# Procesamiento de audio
# ==============================
import soundfile as sf                      # lectura/escritura de audio
import librosa                              # procesamiento de audio
import numpy as np                          # operaciones numéricas sobre señal

# ==============================
# Modelos de IA (ASR)
# ==============================
from faster_whisper import WhisperModel     # modelo ASR Whisper optimizado

# ==============================
# Evaluación de modelos
# ==============================
from jiwer import wer, cer, process_words   # métricas WER/CER y alineación

# ==============================
# Manipulación de datos
# ==============================
import pandas as pd                         # datos tabulares
from collections import Counter             # conteo de frecuencias

# ==============================
# Procesamiento de texto
# ==============================
import re                                   # expresiones regulares
import unicodedata                          # normalización de texto
from num2words import num2words             # conversión de números a texto

# ==============================
# Visualización y análisis
# ==============================
import matplotlib.pyplot as plt             # visualización de datos
import seaborn as sns                       # visualización estadística

# ==============================
# Utilidades y control
# ==============================
from tqdm import tqdm                       # barra de progreso
import time                                 # medición de tiempo

### 2.3 Gestión y configuración de rutas

In [11]:
# Detectar raíz
project_root = Path.cwd()

while not (project_root / "data").exists():
    project_root = project_root.parent


# =========================
# DIRECTORIOS
# =========================

# Ruta raíz del proyecto
data_dir = project_root / "data"


# =========================
# CONFIGURACIÓN (DOMINIO)
# =========================

# Ruta al JSON de términos de dominio
domain_terms_path = data_dir / "config" / "domain" / "coffee_cacao_terms.json"


# =========================
# AUDIOS
# =========================

# Carpeta base de audios
audio_dir = data_dir / "audio"

# Baseline (audios estandarizados)
standardized_audio_dir = audio_dir / "standardized"

# Pipeline final (audios procesados)
processed_audio_dir = audio_dir / "processed"

# Audios en crudo
raw_audio_dir = audio_dir / "raw"


# =========================
# TRANSCRIPCIONES
# =========================

# Carpeta general de transcripciones
transcriptions_dir = data_dir / "transcriptions"

# Salida final del ASR
asr_output_dir = transcriptions_dir / "asr_output"

# Predicciones del ASR (experimentos)
predictions_dir = transcriptions_dir / "predictions"

# Ground truth (transcripciones manuales)
ground_truth_dir = transcriptions_dir / "ground_truth"


# =========================
# CREACIÓN DE CARPETAS
# =========================

asr_output_dir.mkdir(parents=True, exist_ok=True)
predictions_dir.mkdir(parents=True, exist_ok=True)
ground_truth_dir.mkdir(parents=True, exist_ok=True)


# =========================
# VERIFICACIÓN
# =========================

print("Ruta raíz:", project_root)

print("\n--- Configuración ---")
print("Términos de dominio:", domain_terms_path)

print("\n--- Audios ---")
print("Raw:", raw_audio_dir)
print("Baseline (standardized):", standardized_audio_dir)
print("Pipeline final (processed):", processed_audio_dir)

print("\n--- Transcripciones ---")
print("Salida ASR:", asr_output_dir)
print("Predicciones (experimentos):", predictions_dir)
print("Ground truth:", ground_truth_dir)

Ruta raíz: /Volumes/EXTENSION/GitHub/TFM

--- Configuración ---
Términos de dominio: /Volumes/EXTENSION/GitHub/TFM/data/config/domain/coffee_cacao_terms.json

--- Audios ---
Raw: /Volumes/EXTENSION/GitHub/TFM/data/audio/raw
Baseline (standardized): /Volumes/EXTENSION/GitHub/TFM/data/audio/standardized
Pipeline final (processed): /Volumes/EXTENSION/GitHub/TFM/data/audio/processed

--- Transcripciones ---
Salida ASR: /Volumes/EXTENSION/GitHub/TFM/data/transcriptions/asr_output
Predicciones (experimentos): /Volumes/EXTENSION/GitHub/TFM/data/transcriptions/predictions
Ground truth: /Volumes/EXTENSION/GitHub/TFM/data/transcriptions/ground_truth


## 3. Preparación del *dataset* para ASR

En esta sección se construye el conjunto de datos de entrada para el modelo ASR a partir de los audios preprocesados en la etapa anterior. Se consideran dos variantes del *dataset*: una basada en audios estandarizados (baseline) y otra correspondiente al *pipeline* completo de preprocesamiento. Además, se incorporan mecanismos de validación para garantizar que las señales cumplen los requisitos técnicos necesarios para su correcta transcripción.

### 3.1 Carga de audios preprocesados

En primer lugar, se cargan las rutas de los audios generados en la etapa de preprocesamiento. Se distinguen dos conjuntos: audios estandarizados, que constituyen la referencia *baseline*, y audios procesados mediante el *pipeline* completo. Esta separación permite analizar posteriormente el impacto del preprocesamiento sobre el rendimiento del sistema ASR.

In [12]:
# Carga de rutas de audios baseline (estandarizados)
audio_paths_standardized = sorted(list(standardized_audio_dir.glob("*.wav")))

# Carga de rutas de audios procesados (pipeline final)
audio_paths_processed = sorted(list(processed_audio_dir.glob("*.wav")))

# Verificación
print(f"Audios baseline (standardized): {len(audio_paths_standardized)}")
print(f"Audios pipeline final (processed): {len(audio_paths_processed)}")

Audios baseline (standardized): 200
Audios pipeline final (processed): 200


### 3.2 Asociación con metadatos

A continuación, se construye una representación estructurada del *dataset*, asociando a cada audio un identificador único y su ruta en el sistema de archivos. Esta estructura facilita el procesamiento posterior y permite mantener la trazabilidad entre los audios y las transcripciones generadas.

In [13]:
# Construcción de dataset baseline (audios estandarizados)
dataset_standardized = [
    {"audio_id": path.stem, "path": path}
    for path in audio_paths_standardized
]

# Construcción de dataset procesado (pipeline final)
dataset_processed = [
    {"audio_id": path.stem, "path": path}
    for path in audio_paths_processed
]

# Conversión a DataFrame para inspección
df_standardized = pd.DataFrame(dataset_standardized)
df_processed = pd.DataFrame(dataset_processed)

pd.DataFrame({
    "standardized": df_standardized["audio_id"].head(),
    "processed": df_processed["audio_id"].head()
})

,standardized,processed
0,AUDIO_00001,AUDIO_00001
1,AUDIO_00002,AUDIO_00002
2,AUDIO_00003,AUDIO_00003
3,AUDIO_00004,AUDIO_00004
4,AUDIO_00005,AUDIO_00005


### 3.3 Validación de entradas para transcripción

Antes de proceder a la transcripción, es necesario verificar que los audios cumplen los requisitos técnicos exigidos por el modelo ASR. Esta validación permite detectar posibles inconsistencias derivadas del proceso de preprocesamiento y evitar errores en etapas posteriores.

#### 3.3.1 Verificación de formato (wav, 16kHz, mono)

Se comprueba que los audios se encuentran en formato WAV, con una frecuencia de muestreo de 16 kHz y en canal mono, condiciones requeridas por el modelo ASR utilizado. Esta verificación garantiza la compatibilidad de las señales de entrada y evita degradaciones en el rendimiento derivadas de formatos no adecuados.

In [14]:
# Validación de audios baseline (estandarizados)
for item in dataset_standardized:
    try:
        info = sf.info(item["path"])
        item["samplerate"] = info.samplerate
        item["channels"] = info.channels
        item["format_ok"] = (info.samplerate == 16000 and info.channels == 1)
    except Exception as e:
        item["format_ok"] = False
        item["error"] = str(e)


# Validación de audios procesados (pipeline final)
for item in dataset_processed:
    try:
        info = sf.info(item["path"])
        item["samplerate"] = info.samplerate
        item["channels"] = info.channels
        item["format_ok"] = (info.samplerate == 16000 and info.channels == 1)
    except Exception as e:
        item["format_ok"] = False
        item["error"] = str(e)

#### 3.3.2 Control de errores y audios inválidos

Se identifican aquellos audios que no cumplen los requisitos de formato o presentan errores durante su lectura. En lugar de eliminarlos automáticamente, se separan en conjuntos de válidos e inválidos, permitiendo analizar su impacto y mantener la trazabilidad del proceso.

In [15]:
# Filtrado baseline (audios estandarizados)
valid_standardized = [item for item in dataset_standardized if item["format_ok"]]
invalid_standardized = [item for item in dataset_standardized if not item["format_ok"]]

print(f"Standardized → válidos: {len(valid_standardized)}, inválidos: {len(invalid_standardized)}")


# Filtrado pipeline final (audios procesados)
valid_processed = [item for item in dataset_processed if item["format_ok"]]
invalid_processed = [item for item in dataset_processed if not item["format_ok"]]

print(f"Processed → válidos: {len(valid_processed)}, inválidos: {len(invalid_processed)}")


# Mostrar errores si existen
if invalid_standardized or invalid_processed:
    print("\nEjemplos de errores:")
    
    for item in (invalid_standardized + invalid_processed)[:5]:
        print(f"- {item['audio_id']}: {item.get('error', 'Formato incorrecto')}")

Standardized → válidos: 200, inválidos: 0
Processed → válidos: 200, inválidos: 0


Este control permite garantizar que únicamente los audios válidos son utilizados en la fase de transcripción, evitando errores en la ejecución del modelo ASR y asegurando la coherencia del pipeline.

## 4. Configuración del modelo ASR

En esta sección se define la configuración del modelo ASR utilizado para la generación de transcripciones. Se detallan tanto la selección del modelo base como los parámetros de inferencia empleados, estableciendo una configuración de referencia (*baseline*) que servirá como punto de partida para la evaluación experimental. Las decisiones adoptadas en esta fase tienen un impacto directo en la calidad, estabilidad y eficiencia del sistema.

### 4.1 Selección del modelo base (Whisper)

Se selecciona el modelo Whisper en su implementación optimizada (*faster-whisper*), ampliamente utilizado por su robustez en entornos reales y su capacidad para manejar variabilidad en las condiciones de audio.

En particular, se emplea el modelo de tamaño medium, que ofrece un equilibrio adecuado entre precisión y coste computacional. Modelos de menor tamaño presentan limitaciones en calidad de transcripción, mientras que versiones más grandes incrementan significativamente el tiempo de inferencia sin aportar mejoras proporcionales en este contexto.

La ejecución se realiza en CPU utilizando cuantización a int8, lo que permite reducir el consumo de memoria y el tiempo de procesamiento, manteniendo un rendimiento adecuado para el análisis experimental.

In [16]:
# Selección del tamaño del modelo
model_size = "medium"  # opciones: tiny, base, small, medium, large-v3

# Configuración de dispositivo (ajusta según tu equipo)
device = "cpu"   # "cuda" si tienes GPU
compute_type = "int8"  # "float16" en GPU, "int8" en CPU

# Carga del modelo
model = WhisperModel(model_size, device=device, compute_type=compute_type)

print(f"Modelo cargado: {model_size}")

Modelo cargado: medium


### 4.2 Parámetros de inferencia

Los parámetros de inferencia se configuran con el objetivo de garantizar consistencia y reproducibilidad en los resultados. Se prioriza un comportamiento determinista del modelo, reduciendo la variabilidad entre ejecuciones y evitando la propagación de errores entre segmentos de audio.

In [17]:
asr_params = {
    "language": "es",                       # fuerza el idioma español, evitando autodetección y posibles errores de idioma
    "beam_size": 1,                         # número de hipótesis evaluadas; 1 = decodificación greedy (rápida y casi determinista)
    "condition_on_previous_text": False,    # evita usar contexto previo entre segmentos, reduciendo propagación de errores
    "word_timestamps": False,               # desactiva timestamps por palabra (innecesario si solo quieres texto)
    "temperature": 0.0                      # desactiva aleatoriedad para resultados consistentes
}

### 4.3 Definición de configuración *baseline*

Con el fin de estructurar el experimento y facilitar la reproducibilidad, se define una configuración *baseline* que agrupa tanto las características del modelo como los parámetros de inferencia. Esta configuración se utilizará como referencia en las fases posteriores de transcripción y evaluación, permitiendo comparar su comportamiento frente a posibles variaciones.

In [18]:
# Configuración baseline utilizada en la transcripción

baseline_config = {
    "model_size": model_size,
    "device": device,
    "compute_type": compute_type,
    "asr_params": asr_params
}

print("Configuración baseline:")
display(baseline_config)

Configuración baseline:


{'model_size': 'medium',
 'device': 'cpu',
 'compute_type': 'int8',
 'asr_params': {'language': 'es',
  'beam_size': 1,
  'condition_on_previous_text': False,
  'word_timestamps': False,
  'temperature': 0.0}}

La definición explícita de esta configuración permite desacoplar la lógica experimental de la implementación, facilitando la exploración sistemática de distintas alternativas sin alterar la estructura del *pipeline*.

## 5. Transcripción automática del audio

En esta sección se aborda la conversión de los audios preprocesados en representaciones textuales mediante un sistema ASR. Se evalúan dos configuraciones diferenciadas: una aproximación *baseline* basada en audios estandarizados, y un *pipeline* completo que incorpora técnicas de mejora de señal. El objetivo es analizar el impacto del preprocesamiento en la calidad de las transcripciones generadas, que serán utilizadas posteriormente en la evaluación cuantitativa y en las tareas de procesamiento del lenguaje natural.

### 5.1 Generación de transcripciones (*baseline*)

En este apartado se generan las transcripciones automáticas a partir de los audios procesados en etapas anteriores. Se consideran dos conjuntos de entrada: audios estandarizados (*baseline*) y audios sometidos al *pipeline* completo de preprocesamiento. En ambos casos se utiliza el mismo modelo ASR y configuración de inferencia, permitiendo una comparación directa del efecto del procesamiento previo sobre la calidad del texto generado.

In [19]:
# Transcripción baseline (audios estandarizados)
results_standardized = []

for item in tqdm(dataset_standardized):
    segments, _ = model.transcribe(
        str(item["path"]),
        **asr_params
    )
    
    text = " ".join([seg.text for seg in segments]).strip()
    
    results_standardized.append({
        "audio_id": item["audio_id"],
        "text": text
    })


# Transcripción pipeline final (audios procesados)
results_processed = []

for item in tqdm(dataset_processed):
    segments, _ = model.transcribe(
        str(item["path"]),
        **asr_params
    )
    
    text = " ".join([seg.text for seg in segments]).strip()
    
    results_processed.append({
        "audio_id": item["audio_id"],
        "text": text
    })

  0%|          | 0/200 [00:00<?, ?it/s]

100%|██████████| 200/200 [15:05<00:00,  4.53s/it]


El resultado de este proceso es una lista de transcripciones estructuradas, donde cada elemento contiene el identificador del audio y el texto generado. Estas estructuras servirán como base para el almacenamiento, la validación y el cálculo de métricas en las siguientes etapas.

### 5.2 Almacenamiento estructurado de resultados

Antes de ejecutar este bloque, se recomienda limpiar manualmente los directorios de salida para evitar inconsistencias entre ejecuciones.

In [20]:
# Guardado baseline (audios estandarizados)
for item in results_standardized:
    output_path = predictions_dir / "standardized" / f"{item['audio_id']}.json"
    
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(item, f, ensure_ascii=False, indent=2)


# Guardado pipeline final (audios procesados)
for item in results_processed:
    output_path = predictions_dir / "processed" / f"{item['audio_id']}.json"
    
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(item, f, ensure_ascii=False, indent=2)

Las transcripciones generadas se almacenan en formato JSON, donde cada archivo representa un audio procesado. La estructura contiene el identificador del audio (`audio_id`) y el texto transcrito (`text`), facilitando su posterior uso en el cálculo de métricas y en las etapas de procesamiento del lenguaje natural.

Ejemplo:

```json
{
  "audio_id": "AUDIO_00001",
  "text": "aplique fungicida en lote 3"
}
```

Este formato permite desacoplar la fase de generación de transcripciones de las etapas posteriores del pipeline, facilitando la reproducibilidad del experimento y el análisis independiente de los resultados.

### 5.3 Verificación de transcripciones generadas

Se realiza una verificación básica de las transcripciones generadas con el objetivo de inspeccionar su estructura y contenido. Esta comprobación permite detectar errores evidentes en la generación del texto antes de proceder a su almacenamiento o evaluación.

In [21]:
# Ejemplo de verificación

if results_standardized:
    print(results_standardized[0])

if results_processed:
    print(results_processed[0])

{'audio_id': 'AUDIO_00001', 'text': 'Bueno, hoy estuvimos haciendo deshierbe en el lote porque estaba muy amontonado y eso afecta bastante al desarrollo del café.'}
{'audio_id': 'AUDIO_00001', 'text': 'Bueno, hoy estuvimos haciendo deshierbe en el lote porque estaba muy amontonado y eso afecta bastante al desarrollo del café.'}


### 5.4 Control de errores

Dado que los sistemas ASR pueden generar transcripciones vacías en determinados casos (por ejemplo, audios con baja calidad o ausencia de voz), se incorpora un mecanismo de control para identificar estas situaciones. En lugar de eliminar estos registros, se detectan y reportan para su análisis, manteniendo la integridad de los datos originales y permitiendo su exclusión controlada en etapas posteriores del *pipeline*.

In [22]:
# Detección de textos vacíos en baseline
empty_standardized = [r for r in results_standardized if not r["text"].strip()]

# Detección de textos vacíos en pipeline final
empty_processed = [r for r in results_processed if not r["text"].strip()]

print(f"Baseline → textos vacíos: {len(empty_standardized)}")
print(f"Processed → textos vacíos: {len(empty_processed)}")

# Mostrar ejemplos si existen
if empty_standardized or empty_processed:
    print("\nEjemplos de audios con transcripción vacía:")
    
    for r in (empty_standardized + empty_processed)[:5]:
        print(f"- {r['audio_id']}")

Baseline → textos vacíos: 0
Processed → textos vacíos: 0


Este control permite cuantificar la incidencia de fallos en la transcripción sin comprometer la trazabilidad del sistema, diferenciando entre datos generados y datos efectivamente utilizables en tareas posteriores.

## 6. Evaluación del rendimiento del ASR

Intro aqui

### 6.1 Definición del conjunto de referencia (*ground truth*)

Intro aqui

In [23]:
# Carga del archivo CSV de ground truth
df_gt = pd.read_csv(ground_truth_dir / "ground_truth.csv")

# Eliminamos posibles filas con valores nulos
df_gt = df_gt.dropna(subset=["audio_id", "transcripcion"])

# Verificamos que no existan identificadores duplicados
assert df_gt["audio_id"].is_unique, "Error: existen audio_id duplicados en el ground truth"

# Construcción del diccionario {audio_id: transcripcion}
ground_truth = dict(zip(df_gt["audio_id"], df_gt["transcripcion"]))

print(f"Total ground truth: {len(ground_truth)}")

Total ground truth: 200


### 6.2 Normalización previa de texto

Antes de calcular las métricas de evaluación, se aplica una normalización controlada sobre las transcripciones de referencia (*ground truth*) y las generadas por el modelo ASR.

Esta normalización tiene como objetivo eliminar diferencias superficiales que no reflejan errores reales de reconocimiento, como variaciones en capitalización, formato numérico, acentuación o puntuación. De este modo, se garantiza que las métricas WER y CER evalúan únicamente discrepancias relevantes en el contenido lingüístico, evitando penalizaciones derivadas de diferencias de formato.

Es importante destacar que esta normalización se aplica exclusivamente en la fase de evaluación, manteniendo las transcripciones originales sin modificar para su uso en etapas posteriores del pipeline.

In [24]:
# Función para normalizar texto antes de calcular métricas ASR
def normalize_text(text):
    
    # Conversión a minúsculas
    text = text.lower()
    
    # Conversión de números enteros a texto
    def replace_number(match):
        try:
            num = int(match.group())
            return num2words(num, lang='es')
        except:
            return match.group()
    
    text = re.sub(r"\b\d+\b", replace_number, text)
    
    # Eliminación de acentos
    text = ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )
    
    # Eliminación de puntuación
    text = re.sub(r"[^\w\s]", "", text)
    
    # Normalización de espacios
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

### 6.3 Cálculo de métricas

En esta sección se evalúa la calidad de las transcripciones generadas por el sistema ASR mediante métricas estándar ampliamente utilizadas en reconocimiento del habla. En particular, se emplean *Word Error Rate* (WER) y *Character Error Rate* (CER), que permiten cuantificar el grado de similitud entre las transcripciones generadas y las referencias manuales.

Estas métricas se calculan a nivel individual para cada audio, lo que permite no solo obtener valores agregados, sino también analizar la distribución del error y detectar comportamientos específicos del sistema en distintos casos.

#### 6.3.1 *Word Error Rate* (WER)

El **WER** mide la proporción de errores a nivel de palabra entre la transcripción generada y la referencia, considerando operaciones de sustitución, inserción y eliminación. Se trata de la métrica más utilizada en sistemas ASR, ya que proporciona una estimación directa del error percibido en el texto.

In [25]:
# Función para calcular el Word Error Rate (WER) para un conjunto de resultados
def compute_wer(results, ground_truth):
    
    # Lista donde se almacenan los valores individuales de WER por audio
    scores = []
    
    # Iteramos sobre cada transcripción generada por el ASR
    for item in results:
        
        # Identificador único del audio
        audio_id = item["audio_id"]
        
        # Verificamos que exista referencia en el conjunto ground truth
        if audio_id in ground_truth:
            
            # Texto de referencia (transcripción manual)
            ref = ground_truth[audio_id]
            
            # Texto predicho por el modelo ASR
            hyp = item["text"]
            
            # Aplicamos la normalización para eliminar diferencias no relevantes
            ref = normalize_text(ref)
            hyp = normalize_text(hyp)
            
            # Evitamos casos vacíos tras normalización
            if ref and hyp:
                
                # Cálculo del WER entre referencia y predicción
                score = wer(ref, hyp)
                
                # Almacenamiento del resultado individual
                scores.append(score)
    
    # Devolvemos la lista completa de errores
    return scores

#### 6.3.2 *Character Error Rate* (CER)

El **CER** complementa el análisis anterior evaluando el error a nivel de carácter. Esta métrica resulta especialmente útil en casos donde pequeñas variaciones ortográficas pueden afectar significativamente al contenido, proporcionando una visión más fina del comportamiento del sistema.

In [26]:
# Función para calcular el Character Error Rate (CER) para un conjunto de resultados
def compute_cer(results, ground_truth):
    
    # Lista donde se almacenan los valores individuales de CER por audio
    scores = []
    
    # Iteramos sobre cada transcripción generada por el ASR
    for item in results:
        
        # Identificador único del audio
        audio_id = item["audio_id"]
        
        # Verificamos que exista referencia en el conjunto ground truth
        if audio_id in ground_truth:
            
            # Texto de referencia (transcripción manual)
            ref = ground_truth[audio_id]
            
            # Texto predicho por el modelo ASR
            hyp = item["text"]
            
            # Aplicamos la misma normalización que en WER para mantener coherencia
            ref = normalize_text(ref)
            hyp = normalize_text(hyp)
            
            # Evitamos casos vacíos tras normalización
            if ref and hyp:
                
                # Cálculo del CER entre referencia y predicción
                score = cer(ref, hyp)
                
                # Almacenamiento del resultado individual
                scores.append(score)
    
    # Devolvemos la lista completa de errores
    return scores

### 6.4 Evaluación cuantitativa

In [27]:
# =========================
# CÁLCULO DE MÉTRICAS
# =========================

# Cálculo explícito (base del análisis)
wer_standardized = compute_wer(results_standardized, ground_truth)
wer_processed = compute_wer(results_processed, ground_truth)

cer_standardized = compute_cer(results_standardized, ground_truth)
cer_processed = compute_cer(results_processed, ground_truth)


# =========================
# DATASET BASE
# =========================

# Identificadores
ids_std = [r["audio_id"] for r in results_standardized]

# Construcción del DataFrame base
df_results = pd.DataFrame({
    "audio_id": ids_std,
    "wer_standardized": wer_standardized,
    "wer_processed": wer_processed,
    "cer_standardized": cer_standardized,
    "cer_processed": cer_processed
})


# =========================
# ESTADÍSTICAS
# =========================

# Calcula estadísticas descriptivas robustas
def compute_robust_stats(values: list[float]) -> dict:
    
    arr = np.array(values)
    arr_sorted = np.sort(arr)
    
    n = len(arr_sorted)
    k = int(0.1 * n)
    
    trimmed = arr_sorted[k:n-k] if n > 2 * k else arr_sorted
    
    return {
        "mean": np.mean(arr),
        "median": np.median(arr),
        "std": np.std(arr),
        "min": np.min(arr),
        "max": np.max(arr),
        "p90": np.percentile(arr, 90),
        "p95": np.percentile(arr, 95),
        "trimmed_mean": np.mean(trimmed)
    }


# =========================
# APLICACIÓN DE ESTADÍSTICAS
# =========================

stats_wer_standardized = compute_robust_stats(df_results["wer_standardized"])
stats_wer_processed = compute_robust_stats(df_results["wer_processed"])

stats_cer_standardized = compute_robust_stats(df_results["cer_standardized"])
stats_cer_processed = compute_robust_stats(df_results["cer_processed"])


# =========================
# VISUALIZACIÓN
# =========================

df_stats = pd.DataFrame([
    {"metric": "WER", "pipeline": "standardized", **stats_wer_standardized},
    {"metric": "WER", "pipeline": "processed", **stats_wer_processed},
    {"metric": "CER", "pipeline": "standardized", **stats_cer_standardized},
    {"metric": "CER", "pipeline": "processed", **stats_cer_processed},
])

display(df_stats)

,metric,pipeline,mean,median,std,min,max,p90,p95,trimmed_mean
0,WER,standardized,0.038824,0.0,0.053510,0.0,0.277778,0.115611,0.142857,0.028705
1,WER,processed,0.037864,0.0,0.053308,0.0,0.277778,0.111538,0.142857,0.027573
2,CER,standardized,0.014586,0.0,0.024328,0.0,0.175000,0.042870,0.058832,0.009497
3,CER,processed,0.013790,0.0,0.023362,0.0,0.175000,0.040596,0.054381,0.009055


Los resultados obtenidos muestran un comportamiento muy consistente del sistema ASR en ambas configuraciones evaluadas. En términos de *Word Error Rate* (WER), se observa una ligera mejora en el *pipeline* procesado respecto al baseline, con una reducción del error medio de aproximadamente 0.039 a 0.038.

Esta mejora, aunque cuantitativamente pequeña, es estable y coherente en el resto de métricas robustas, como la media recortada (*trimmed mean*), que también presenta una disminución.

La mediana en ambos casos es igual a 0.0, lo que indica que más del 50% de las transcripciones no presentan errores a nivel de palabra. Este resultado sugiere un rendimiento elevado del modelo en condiciones habituales, concentrándose los errores en un subconjunto reducido de audios.

El análisis de la dispersión muestra valores prácticamente idénticos entre ambas configuraciones, con desviaciones estándar similares, lo que indica que el preprocesamiento no introduce variabilidad adicional en el comportamiento del sistema. Sin embargo, en los percentiles superiores (P90 y P95) se mantiene una ligera mejora en el *pipeline* procesado, lo que sugiere una reducción marginal en los casos más desfavorables.

En cuanto al *Character Error Rate* (CER), se observa el mismo patrón: el *pipeline* procesado presenta un menor error medio, junto con mejoras consistentes en los percentiles altos y en la media recortada.

El valor máximo de error se mantiene constante en ambas configuraciones, lo que indica que el preprocesamiento no elimina completamente los casos más problemáticos, sino que su impacto se concentra en la mejora del comportamiento medio del sistema.

En conjunto, los resultados evidencian que el *pipeline* de preprocesamiento aporta mejoras consistentes pero de magnitud reducida, mejorando ligeramente la calidad global de las transcripciones sin introducir efectos negativos en la estabilidad del sistema.

### 6.4 Análisis de resultados globales

In [28]:
# =========================
# VALIDACIÓN DE CONSISTENCIA
# =========================

# Verificamos que los audio_id estén alineados entre ambos pipelines
ids_std = [r["audio_id"] for r in results_standardized]
ids_proc = [r["audio_id"] for r in results_processed]

assert ids_std == ids_proc, "Error: los audio_id no están alineados"


# =========================
# ENRIQUECIMIENTO DEL DATASET
# =========================

# Diferencia de error entre pipelines (processed vs standardized)
df_results["delta_wer"] = df_results["wer_processed"] - df_results["wer_standardized"]
df_results["delta_cer"] = df_results["cer_processed"] - df_results["cer_standardized"]


# =========================
# CLASIFICACIÓN DE RESULTADOS
# =========================

# Clasificación del impacto del procesamiento
df_results["wer_estado"] = df_results["delta_wer"].apply(
    lambda x: "mejora" if x < 0 else ("empeora" if x > 0 else "igual")
)

df_results["cer_estado"] = df_results["delta_cer"].apply(
    lambda x: "mejora" if x < 0 else ("empeora" if x > 0 else "igual")
)


# =========================
# RESUMEN GLOBAL
# =========================

wer_summary = df_results["wer_estado"].value_counts(normalize=True) * 100
cer_summary = df_results["cer_estado"].value_counts(normalize=True) * 100

print("Distribución WER:")
for estado, valor in wer_summary.items():
    print(f"{estado}: {valor:.2f}%")

print("\nDistribución CER:")
for estado, valor in cer_summary.items():
    print(f"{estado}: {valor:.2f}%")


# =========================
# ANÁLISIS DE CASOS
# =========================

# Mejores y peores casos (WER)
df_best_wer = df_results[df_results["wer_estado"] == "mejora"].sort_values(by="delta_wer")
df_worst_wer = df_results[df_results["wer_estado"] == "empeora"].sort_values(by="delta_wer", ascending=False)

print("\nTop mejoras (WER):")
display(df_best_wer.head(10))

print("\nPeores casos (WER):")
display(df_worst_wer.head(10))


# Mejores y peores casos (CER)
df_best_cer = df_results[df_results["cer_estado"] == "mejora"].sort_values(by="delta_cer")
df_worst_cer = df_results[df_results["cer_estado"] == "empeora"].sort_values(by="delta_cer", ascending=False)

print("\nTop mejoras (CER):")
display(df_best_cer.head(10))

print("\nPeores casos (CER):")
display(df_worst_cer.head(10))


# =========================
# VISIÓN GENERAL
# =========================

print("\nVisión general:")
display(df_results)

Distribución WER:
igual: 88.00%
mejora: 7.00%
empeora: 5.00%

Distribución CER:
igual: 86.50%
mejora: 8.00%
empeora: 5.50%

Top mejoras (WER):


,audio_id,wer_standardized,wer_processed,cer_standardized,cer_processed,delta_wer,delta_cer,wer_estado,cer_estado
179,AUDIO_00180,0.200000,0.100000,0.072917,0.023438,-0.100000,-0.049479,mejora,mejora
120,AUDIO_00121,0.120000,0.020000,0.100358,0.010753,-0.100000,-0.089606,mejora,mejora
74,AUDIO_00075,0.066667,0.000000,0.021739,0.000000,-0.066667,-0.021739,mejora,mejora
156,AUDIO_00157,0.064815,0.009259,0.019169,0.004792,-0.055556,-0.014377,mejora,mejora
126,AUDIO_00127,0.105263,0.052632,0.024000,0.008000,-0.052632,-0.016000,mejora,mejora
128,AUDIO_00129,0.050000,0.000000,0.014286,0.000000,-0.050000,-0.014286,mejora,mejora
87,AUDIO_00088,0.038462,0.000000,0.006024,0.000000,-0.038462,-0.006024,mejora,mejora
39,AUDIO_00040,0.038168,0.000000,0.009284,0.000000,-0.038168,-0.009284,mejora,mejora
127,AUDIO_00128,0.051813,0.015544,0.016983,0.004995,-0.036269,-0.011988,mejora,mejora
173,AUDIO_00174,0.050847,0.016949,0.022857,0.008571,-0.033898,-0.014286,mejora,mejora



Peores casos (WER):


,audio_id,wer_standardized,wer_processed,cer_standardized,cer_processed,delta_wer,delta_cer,wer_estado,cer_estado
103,AUDIO_00104,0.000000,0.083333,0.000000,0.011236,0.083333,0.011236,empeora,empeora
124,AUDIO_00125,0.104167,0.166667,0.036424,0.062914,0.062500,0.026490,empeora,empeora
3,AUDIO_00004,0.000000,0.052632,0.000000,0.008475,0.052632,0.008475,empeora,empeora
107,AUDIO_00108,0.000000,0.050000,0.000000,0.022901,0.050000,0.022901,empeora,empeora
54,AUDIO_00055,0.047619,0.095238,0.025000,0.050000,0.047619,0.025000,empeora,empeora
79,AUDIO_00080,0.022222,0.066667,0.003460,0.013841,0.044444,0.010381,empeora,empeora
144,AUDIO_00145,0.040000,0.080000,0.019481,0.032468,0.040000,0.012987,empeora,empeora
147,AUDIO_00148,0.000000,0.040000,0.000000,0.020548,0.040000,0.020548,empeora,empeora
130,AUDIO_00131,0.142857,0.178571,0.030488,0.036585,0.035714,0.006098,empeora,empeora
81,AUDIO_00082,0.082353,0.105882,0.042991,0.041121,0.023529,-0.001869,empeora,mejora



Top mejoras (CER):


,audio_id,wer_standardized,wer_processed,cer_standardized,cer_processed,delta_wer,delta_cer,wer_estado,cer_estado
120,AUDIO_00121,0.120000,0.020000,0.100358,0.010753,-0.100000,-0.089606,mejora,mejora
179,AUDIO_00180,0.200000,0.100000,0.072917,0.023438,-0.100000,-0.049479,mejora,mejora
74,AUDIO_00075,0.066667,0.000000,0.021739,0.000000,-0.066667,-0.021739,mejora,mejora
190,AUDIO_00191,0.123967,0.090909,0.058706,0.039360,-0.033058,-0.019346,mejora,mejora
119,AUDIO_00120,0.052632,0.026316,0.022901,0.003817,-0.026316,-0.019084,mejora,mejora
184,AUDIO_00185,0.181818,0.181818,0.051724,0.034483,0.000000,-0.017241,igual,mejora
126,AUDIO_00127,0.105263,0.052632,0.024000,0.008000,-0.052632,-0.016000,mejora,mejora
156,AUDIO_00157,0.064815,0.009259,0.019169,0.004792,-0.055556,-0.014377,mejora,mejora
128,AUDIO_00129,0.050000,0.000000,0.014286,0.000000,-0.050000,-0.014286,mejora,mejora
173,AUDIO_00174,0.050847,0.016949,0.022857,0.008571,-0.033898,-0.014286,mejora,mejora



Peores casos (CER):


,audio_id,wer_standardized,wer_processed,cer_standardized,cer_processed,delta_wer,delta_cer,wer_estado,cer_estado
124,AUDIO_00125,0.104167,0.166667,0.036424,0.062914,0.062500,0.026490,empeora,empeora
54,AUDIO_00055,0.047619,0.095238,0.025000,0.050000,0.047619,0.025000,empeora,empeora
107,AUDIO_00108,0.000000,0.050000,0.000000,0.022901,0.050000,0.022901,empeora,empeora
147,AUDIO_00148,0.000000,0.040000,0.000000,0.020548,0.040000,0.020548,empeora,empeora
144,AUDIO_00145,0.040000,0.080000,0.019481,0.032468,0.040000,0.012987,empeora,empeora
103,AUDIO_00104,0.000000,0.083333,0.000000,0.011236,0.083333,0.011236,empeora,empeora
79,AUDIO_00080,0.022222,0.066667,0.003460,0.013841,0.044444,0.010381,empeora,empeora
3,AUDIO_00004,0.000000,0.052632,0.000000,0.008475,0.052632,0.008475,empeora,empeora
11,AUDIO_00012,0.060606,0.060606,0.012500,0.020833,0.000000,0.008333,igual,empeora
150,AUDIO_00151,0.086957,0.086957,0.022059,0.029412,0.000000,0.007353,igual,empeora



Visión general:


,audio_id,wer_standardized,wer_processed,cer_standardized,cer_processed,delta_wer,delta_cer,wer_estado,cer_estado
0,AUDIO_00001,0.142857,0.142857,0.064000,0.064000,0.000000,0.000000,igual,igual
1,AUDIO_00002,0.019608,0.019608,0.002941,0.002941,0.000000,0.000000,igual,igual
2,AUDIO_00003,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,igual,igual
3,AUDIO_00004,0.000000,0.052632,0.000000,0.008475,0.052632,0.008475,empeora,empeora
4,AUDIO_00005,0.068182,0.068182,0.019011,0.019011,0.000000,0.000000,igual,igual
...,...,...,...,...,...,...,...,...,...
195,AUDIO_00196,0.090909,0.090909,0.041096,0.041096,0.000000,0.000000,igual,igual
196,AUDIO_00197,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,igual,igual
197,AUDIO_00198,0.037037,0.037037,0.019108,0.019108,0.000000,0.000000,igual,igual
198,AUDIO_00199,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,igual,igual


El análisis de resultados a nivel individual muestra que el impacto del *pipeline* de preprocesamiento sobre la calidad de las transcripciones es limitado en términos globales, pero significativo en casos concretos.

En primer lugar, la distribución de estados indica que la mayoría de los audios no experimentan cambios entre ambas configuraciones. En el caso de WER, un 88.00% de los audios permanecen iguales, mientras que solo un 7.00% mejora y un 5.00% empeora. Este mismo patrón se reproduce en CER, con un 86.50% de casos sin variación, un 8.00% de mejora y un 5.50% de empeoramiento.

Este comportamiento sugiere que el modelo ASR ya presenta un rendimiento elevado sobre la mayoría de los audios, por lo que el margen de mejora mediante preprocesamiento es reducido en términos generales. Sin embargo, el análisis detallado revela que el impacto del *pipeline* no es uniforme.

Los casos de mejora muestran reducciones significativas del error, con disminuciones de hasta 0.10 en WER y mejoras aún más pronunciadas en CER. Estos resultados indican que el preprocesamiento resulta especialmente efectivo en audios con mayor nivel de dificultad, donde la calidad de la señal influye de forma más crítica en la transcripción.

Por el contrario, los casos de empeoramiento se caracterizan principalmente por audios que inicialmente presentaban error nulo o muy bajo, en los que pequeñas alteraciones introducidas por el preprocesamiento generan errores adicionales. Este efecto es especialmente visible en ejemplos donde el WER pasa de 0.0 a valores moderados, evidenciando que el *pipeline* puede introducir ligeras distorsiones en señales que ya eran óptimas.

Cabe destacar que, en la mayoría de los casos negativos, el incremento del error es inferior a las mejoras observadas en los casos positivos, lo que sugiere que el impacto global del preprocesamiento sigue siendo favorable.

El análisis comparativo entre WER y CER muestra además una alta coherencia en el comportamiento del sistema. Los audios que mejoran en WER tienden también a mejorar en CER, lo que indica que las mejoras no se limitan a nivel superficial, sino que reflejan una mejor alineación general entre la transcripción generada y la referencia.

En conjunto, estos resultados evidencian que el *pipeline* de preprocesamiento no produce cambios significativos en la mayoría de los casos, pero aporta mejoras relevantes en un subconjunto específico de audios más complejos, a costa de introducir ligeros empeoramientos en casos donde el modelo ya funcionaba correctamente.

Este comportamiento sugiere que el impacto del preprocesamiento es selectivo y dependiente de las características del audio, sin evidenciar una mejora global uniforme, pero sí una optimización en escenarios más exigentes.

## 7. Análisis de errores de transcripción

En esta sección se realiza un análisis detallado de los errores generados por el sistema ASR, con el objetivo de caracterizar su naturaleza y comprender su impacto sobre la calidad de las transcripciones.

A diferencia de las métricas agregadas, que proporcionan una visión global del rendimiento, este análisis permite identificar patrones recurrentes de error y evaluar cómo afectan a distintos tipos de contenido. Este enfoque resulta especialmente relevante para interpretar el comportamiento del sistema en escenarios reales y anticipar su impacto en tareas posteriores de procesamiento del lenguaje natural.

### 7.1 Identificación de errores frecuentes

Se identifican los tipos de error más frecuentes a partir del alineamiento entre las transcripciones generadas y las referencias manuales. Para ello, se analizan tres categorías fundamentales: sustituciones, inserciones y eliminaciones, que corresponden a las operaciones básicas utilizadas en el cálculo de WER.

Este análisis permite descomponer el error global en componentes interpretables, facilitando la detección de patrones sistemáticos en el comportamiento del modelo.

#### 7.1.1 Sustituciones

Las sustituciones representan casos en los que una palabra de la referencia es reemplazada por otra en la transcripción generada. Este tipo de error es especialmente relevante, ya que puede implicar una alteración directa del significado del mensaje.

Para su análisis, se consideran únicamente sustituciones uno a uno (palabra a palabra), con el fin de evitar ambigüedades derivadas de alineamientos complejos y centrarse en errores directamente interpretables.

In [29]:
# Función para obtener las sustituciones más frecuentes
def get_substitutions(results, ground_truth):
    
    subs = Counter()
    
    for item in results:
        audio_id = item["audio_id"]
        
        if audio_id in ground_truth:
            
            # Normalización coherente con WER/CER
            ref = normalize_text(ground_truth[audio_id])
            hyp = normalize_text(item["text"])
            
            if ref and hyp:
                
                ref_words = ref.split()
                hyp_words = hyp.split()
                
                alignment = process_words(ref, hyp)
                
                for group in alignment.alignments:
                    for chunk in group:
                        
                        if chunk.type == "substitute":
                            
                            ref_segment = ref_words[chunk.ref_start_idx:chunk.ref_end_idx]
                            hyp_segment = hyp_words[chunk.hyp_start_idx:chunk.hyp_end_idx]
                            
                            # FILTRO CLAVE: solo sustituciones 1 a 1 (palabra a palabra)
                            if len(ref_segment) == 1 and len(hyp_segment) == 1:
                                
                                ref_word = ref_segment[0]
                                hyp_word = hyp_segment[0]
                                
                                subs[(ref_word, hyp_word)] += 1
    
    return subs


# Cálculo
subs_standardized = get_substitutions(results_standardized, ground_truth)
subs_processed = get_substitutions(results_processed, ground_truth)


# Visualización
print("Sustituciones más frecuentes (filtradas):")

print("\nStandardized:")
for (ref, hyp), count in subs_standardized.most_common(10):
    print(f"{ref} → {hyp} ({count})")

print("\nProcessed:")
for (ref, hyp), count in subs_processed.most_common(10):
    print(f"{ref} → {hyp} ({count})")

Sustituciones más frecuentes (filtradas):

Standardized:
arabica → arabicas (4)
hectareas → tareas (3)
del → de (2)
atehortua → atortua (2)
geisha → heysa (2)
moyobamba → bamba (2)
typica → tipicas (2)
enmontado → amontonado (1)
el → al (1)
pitalito → patalito (1)

Processed:
arabica → arabicas (4)
hectareas → tareas (3)
del → de (2)
atehortua → atortua (2)
criollo → crio (2)
moyobamba → bamba (2)
typica → tipicas (2)
enmontado → amontonado (1)
el → al (1)
mazorcas → mazurcas (1)


El análisis de sustituciones muestra patrones muy similares en ambas configuraciones, lo que indica que este tipo de error depende principalmente del modelo ASR y no del preprocesamiento.

Se observan tres tipos principales de errores:

* Variaciones morfológicas (*arabica* → *arabicas*, *typica* → *tipicas*), que no alteran significativamente el significado.
* Confusiones fonéticas (*hectareas* → *tareas*, *geisha* → *heysa*), asociadas a similitud acústica.
* Errores en nombres propios o términos específicos (*atehortua* → *atortua*, *moyobamba* → *bamba*), donde el modelo pierde precisión.

También aparecen simplificaciones (*del* → *de*, *criollo* → *crio*), que reflejan pérdida parcial de información.

Las diferencias entre *baseline* y *processed* son mínimas, manteniéndose prácticamente los mismos errores. Esto sugiere que el preprocesamiento no modifica de forma relevante la naturaleza de las sustituciones.

#### 7.1.2 Inserciones

Las inserciones corresponden a palabras adicionales generadas por el modelo ASR que no están presentes en la referencia. Este tipo de error suele estar asociado a ruido, repeticiones o interpretaciones incorrectas de pausas en el audio.

Al igual que en el caso anterior, se restringe el análisis a inserciones de una única palabra para facilitar su interpretación.

In [30]:
# Función para calcular las inserciones más frecuentes
def get_insertions(results, ground_truth):
    
    ins = Counter()  # Contador de inserciones
    
    for item in results:
        audio_id = item["audio_id"]  # Identificador del audio
        
        # Verificamos que exista en el ground truth
        if audio_id in ground_truth:
            
            # Normalización coherente con el resto del análisis
            ref = normalize_text(ground_truth[audio_id])
            hyp = normalize_text(item["text"])
            
            # Evitamos casos vacíos
            if ref and hyp:
                
                # Tokenización simple por palabras
                ref_words = ref.split()
                hyp_words = hyp.split()
                
                # Alineamiento entre referencia y predicción
                alignment = process_words(ref, hyp)
                
                # Recorremos bloques de alineamiento
                for group in alignment.alignments:
                    for chunk in group:
                        
                        # Detectamos inserciones
                        if chunk.type == "insert":
                            
                            # Segmento insertado en la hipótesis
                            hyp_segment = hyp_words[chunk.hyp_start_idx:chunk.hyp_end_idx]
                            
                            # FILTRO: solo palabras individuales (evita ruido)
                            if len(hyp_segment) == 1:
                                
                                hyp_word = hyp_segment[0]  # Palabra insertada
                                
                                ins[hyp_word] += 1  # Contabilizamos
    
    return ins


# =========================
# CÁLCULO
# =========================

# Calculamos inserciones para ambos pipelines
ins_standardized = get_insertions(results_standardized, ground_truth)
ins_processed = get_insertions(results_processed, ground_truth)


# =========================
# VISUALIZACIÓN
# =========================

print("Inserciones más frecuentes (filtradas):")

print("\nStandardized:")
for word, count in ins_standardized.most_common(10):
    print(f"{word} ({count})")

print("\nProcessed:")
for word, count in ins_processed.most_common(10):
    print(f"{word} ({count})")

Inserciones más frecuentes (filtradas):

Standardized:
y (6)
a (4)
el (2)
la (2)
moyo (2)
de (2)
carana (1)
agro (1)
asi (1)
chau (1)

Processed:
a (4)
de (4)
el (3)
y (3)
la (2)
moyo (2)
agro (1)
asi (1)
post (1)
poder (1)


Las inserciones están dominadas principalmente por palabras funcionales como *y*, *a*, *de*, *el* y *la*, lo que indica que el modelo tiende a añadir conectores o artículos que no están presentes en la referencia.

Este tipo de error suele estar asociado a pausas, ambigüedad en la segmentación o interpretaciones incorrectas del ritmo del habla, más que a errores semánticos relevantes.

También aparecen fragmentos parciales (*moyo*, *carana*, *agro*), lo que sugiere problemas en la reconstrucción de palabras completas a partir de la señal acústica.

La comparación entre ambas configuraciones muestra pequeñas variaciones en la frecuencia de algunas inserciones, pero sin cambios estructurales en el patrón de error. Esto indica que el preprocesamiento tiene un impacto limitado en este tipo de errores, manteniéndose la tendencia general del modelo.

#### 7.1.3 Eliminaciones

Las eliminaciones representan palabras presentes en la referencia que no han sido reconocidas por el modelo ASR. Este tipo de error puede implicar pérdida de información relevante, especialmente en contextos donde ciertos términos son clave para la interpretación del mensaje.

El análisis se centra en eliminaciones individuales, permitiendo identificar qué términos tienden a omitirse con mayor frecuencia.

In [31]:
# Función para calcular las eliminaciones más frecuentes
def get_deletions(results, ground_truth):
    
    dels = Counter()
    
    for item in results:
        audio_id = item["audio_id"]
        
        if audio_id in ground_truth:
            
            # Normalización coherente
            ref = normalize_text(ground_truth[audio_id])
            hyp = normalize_text(item["text"])
            
            if ref and hyp:
                
                ref_words = ref.split()
                
                alignment = process_words(ref, hyp)
                
                for group in alignment.alignments:
                    for chunk in group:
                        
                        if chunk.type == "delete":
                            
                            ref_segment = ref_words[chunk.ref_start_idx:chunk.ref_end_idx]
                            
                            # FILTRO: solo palabras individuales
                            if len(ref_segment) == 1:
                                
                                ref_word = ref_segment[0]
                                
                                dels[ref_word] += 1
    
    return dels


# Cálculo
dels_standardized = get_deletions(results_standardized, ground_truth)
dels_processed = get_deletions(results_processed, ground_truth)


# Visualización
print("Eliminaciones más frecuentes (filtradas):")

print("\nStandardized:")
for word, count in dels_standardized.most_common(10):
    print(f"{word} ({count})")

print("\nProcessed:")
for word, count in dels_processed.most_common(10):
    print(f"{word} ({count})")

Eliminaciones más frecuentes (filtradas):

Standardized:
eh (16)
y (4)
en (3)
hectareas (3)
a (2)
de (2)
ya (1)
angel (1)
la (1)
ucayali (1)

Processed:
eh (16)
en (3)
hectareas (3)
y (2)
ya (1)
angel (1)
la (1)
ucayali (1)
a (1)
mixto (1)


Las eliminaciones están claramente dominadas por la interjección *eh*, que aparece con una frecuencia muy superior al resto. Esto indica que el modelo tiende a eliminar muletillas o elementos propios del habla espontánea, lo que sugiere un cierto efecto de “limpieza” del discurso.

También se observan eliminaciones de palabras funcionales (*y*, *en*, *a*, *de*), similares a las insertadas, lo que refleja inestabilidad en la detección de conectores en función del contexto.

En algunos casos aparecen términos relevantes como *hectareas*, lo que indica pérdida de información potencialmente importante, especialmente en contextos de dominio.

La comparación entre ambas configuraciones muestra un patrón prácticamente idéntico, con ligeras variaciones en frecuencia. Esto sugiere que el preprocesamiento no modifica de forma significativa este tipo de error.

#### 7.1.4 Resumen cuantitativo de errores

Con el objetivo de complementar el análisis cualitativo, se construye un resumen cuantitativo que recoge el número total de sustituciones, inserciones y eliminaciones para cada configuración.

Este resumen permite comparar de forma directa el comportamiento global del sistema entre el baseline y el pipeline procesado, proporcionando una visión agregada del tipo de errores predominantes.

In [32]:
# Construcción de tabla resumen
df_error_summary = pd.DataFrame({
    "Pipeline": ["Standardized", "Processed"],
    "Sustituciones": [
        sum(subs_standardized.values()),
        sum(subs_processed.values())
    ],
    "Inserciones": [
        sum(ins_standardized.values()),
        sum(ins_processed.values())
    ],
    "Eliminaciones": [
        sum(dels_standardized.values()),
        sum(dels_processed.values())
    ]
})

# Total
df_error_summary["Total"] = (
    df_error_summary["Sustituciones"] +
    df_error_summary["Inserciones"] +
    df_error_summary["Eliminaciones"]
)

display(df_error_summary)

,Pipeline,Sustituciones,Inserciones,Eliminaciones,Total
0,Standardized,107,33,41,181
1,Processed,100,27,34,161


El análisis cuantitativo muestra una reducción global del número de errores en el *pipeline* procesado respecto al *baseline*. En concreto, el número total de errores pasa de 181 a 161, lo que supone una disminución consistente en todas las categorías analizadas.

Las sustituciones se reducen de 107 a 100, mientras que las inserciones pasan de 33 a 27 y las eliminaciones de 41 a 34. Esta mejora es especialmente notable en inserciones y eliminaciones, lo que indica un mejor ajuste en la segmentación y en la detección de contenido relevante.

Estos resultados confirman que, aunque el impacto del preprocesamiento es limitado en términos globales, sí contribuye a reducir la cantidad total de errores, reforzando la mejora observada en las métricas WER y CER.

#### 7.1.5 Conclusiones de errores de transcripción

El análisis de errores muestra que las sustituciones constituyen el tipo de error predominante, seguidas por eliminaciones e inserciones en menor medida. Este comportamiento indica que el modelo ASR presenta mayores dificultades en la discriminación léxica que en la segmentación del discurso.

Los patrones de error se mantienen prácticamente constantes entre el *baseline* y el *pipeline* procesado, lo que sugiere que la naturaleza de los errores está dominada por las limitaciones del modelo y no por las condiciones de entrada.

No obstante, se observa una reducción consistente en el número total de errores en el *pipeline* procesado, especialmente en inserciones y eliminaciones. Este resultado indica una mejora en la estabilidad de la transcripción, asociada a una mejor segmentación y a una menor generación de contenido adulterado.

En términos prácticos, el preprocesamiento no modifica el tipo de errores que comete el sistema, pero sí reduce su frecuencia, aportando una mejora moderada en la calidad global de las transcripciones.

### 7.2 Análisis de errores con impacto en la estructuración

En esta sección se analizan los errores de transcripción que afectan directamente a la estructuración de la información. A diferencia del análisis anterior, centrado en el error lingüístico general, aquí se estudia el impacto de los errores sobre elementos clave para la extracción de información, como términos de dominio y valores numéricos.

Este enfoque permite evaluar no solo la calidad de la transcripción, sino su utilidad real en tareas posteriores de procesamiento del lenguaje natural.

#### 7.2.1 Términos de dominio

Para evaluar el impacto de los errores sobre el contenido relevante del dominio, se define un conjunto estructurado de términos agrícolas, que incluye cultivos, prácticas, unidades y otros conceptos clave.

Este conjunto se utiliza como referencia para detectar la ausencia de términos relevantes en las transcripciones generadas por el modelo.

Ejemplo:
```json
{
  "cultivo_cafe": ["café", "cafetal"],
  "plaga": ["plaga", "bicho", "insecto"],
  "producto_fitosanitario": ["fungicida", "herbicida", "insecticida", "veneno"],
  "unidad_parcela": ["lote", "parcela", "finca"],
  "cosecha": ["cosecha", "recolección"]
}
```

Los términos de dominio se definen externamente en formato JSON, lo que permite su actualización independiente del código y facilita su reutilización en distintas fases del *pipeline*.

In [33]:
# ==============================
# CARGA DE TÉRMINOS DE DOMINIO
# ==============================

# Cargamos el JSON usando la ruta ya definida en configuración
with open(domain_terms_path, "r", encoding="utf-8") as f:
    domain_json = json.load(f)


# ==============================
# EXTRACCIÓN DE TÉRMINOS
# ==============================

# Convierte el JSON estructurado en una lista plana de términos
def extract_domain_terms(domain_json: dict) -> list[str]:
    
    terms = set()  # Usamos set para evitar duplicados
    
    # Recorremos categorías y términos
    for category in domain_json.values():
        for key, variants in category.items():
            for term in variants:
                terms.add(term)
    
    return list(terms)


# Generamos la lista final de términos
domain_terms = extract_domain_terms(domain_json)


# ==============================
# DETECCIÓN DE ERRORES DE DOMINIO
# ==============================

# Detecta sustituciones en términos de dominio (ref → hyp)
def domain_error_details(
    results: list[dict],
    ground_truth: dict
) -> Counter:
    
    errors = Counter()  # Contador de sustituciones
    
    for item in results:
        audio_id = item["audio_id"]
        
        # Verificamos que exista en el ground truth
        if audio_id in ground_truth:
            
            # Normalización coherente con el resto del análisis
            ref = normalize_text(ground_truth[audio_id])
            hyp = normalize_text(item["text"])
            
            if ref and hyp:
                
                ref_words = ref.split()
                hyp_words = hyp.split()
                
                # Alineamiento palabra a palabra
                alignment = process_words(ref, hyp)
                
                # Recorremos los bloques de alineamiento
                for group in alignment.alignments:
                    for chunk in group:
                        
                        # Nos quedamos solo con sustituciones
                        if chunk.type == "substitute":
                            
                            ref_segment = ref_words[chunk.ref_start_idx:chunk.ref_end_idx]
                            hyp_segment = hyp_words[chunk.hyp_start_idx:chunk.hyp_end_idx]
                            
                            # FILTRO: solo palabra a palabra
                            if len(ref_segment) == 1 and len(hyp_segment) == 1:
                                
                                ref_word = ref_segment[0]
                                hyp_word = hyp_segment[0]
                                
                                # Solo si la palabra original es término de dominio
                                if ref_word in domain_terms:
                                    errors[(ref_word, hyp_word)] += 1
    
    return errors


# ==============================
# CÁLCULO DE ERRORES
# ==============================

domain_errors_std = domain_error_details(results_standardized, ground_truth)
domain_errors_proc = domain_error_details(results_processed, ground_truth)


# ==============================
# VISUALIZACIÓN
# ==============================

print("Sustituciones en términos de dominio:")

print("\nStandardized:")
if domain_errors_std:
    for (ref, hyp), count in domain_errors_std.most_common(10):
        print(f"{ref} → {hyp} ({count})")
else:
    print("sin errores")

print("\nProcessed:")
if domain_errors_proc:
    for (ref, hyp), count in domain_errors_proc.most_common(10):
        print(f"{ref} → {hyp} ({count})")
else:
    print("sin errores")

Sustituciones en términos de dominio:

Standardized:
moniliasis → monidiasis (1)
abono → un (1)
cacao → cacau (1)
fertilizacion → fertilidadion (1)
machete → machetes (1)
cafe → cofe (1)
moniliasis → monileasis (1)
deshierbe → desierve (1)
cacao → kakan (1)
cacao → caca (1)

Processed:
moniliasis → monidiasis (1)
abono → un (1)
cacao → cacau (1)
fertilizacion → fertilidadion (1)
machete → machetes (1)
cafe → caffe (1)
moniliasis → monileasis (1)
poda → podas (1)
cacao → kakan (1)
cacao → caca (1)


Los errores se concentran en términos clave del dominio como *moniliasis*, *cacao* o *deshierbe*, lo que implica una distorsión directa de información relevante para la estructuración.

Se observa que estos errores afectan principalmente a vocabulario específico, lo que confirma que el modelo presenta mayores dificultades en términos menos frecuentes o más especializados.

El análisis de sustituciones muestra patrones consistentes de transformación, como simplificaciones fonéticas (*cacao* → *caca*, *cafe* → *cofe*) o modificaciones morfológicas (*machete* → *machetes*, *poda* → *podas*), lo que indica que el modelo tiende a aproximar términos desconocidos a formas más frecuentes o cercanas fonéticamente.

La comparación entre configuraciones muestra un comportamiento muy similar, con ligeras variaciones en algunos términos (*deshierbe*, *poda*). Esto indica que el preprocesamiento no modifica de forma significativa este tipo de error, aunque puede influir en casos concretos.

Este tipo de error resulta especialmente crítico, ya que, aunque el término no desaparece completamente, su alteración puede impedir su correcta identificación en etapas posteriores del *pipeline*.

#### 7.2.2 Números y cantidades

Los valores numéricos constituyen un elemento crítico en la estructuración de la información, especialmente en contextos agrícolas donde cantidades, superficies o medidas tienen un significado relevante.

En esta sección se analizan los errores asociados a la detección de números, considerando tanto representaciones en formato dígito como en texto.

Se identifican errores cuando un valor numérico presente en la referencia no aparece en la transcripción generada, lo que permite evaluar la pérdida de información cuantitativa.

In [34]:
# ==============================
# EXTRACCIÓN DE NÚMEROS
# ==============================

# Función para extraer números (dígitos y texto)
def extract_numbers(text):
    
    text = normalize_text(text)
    
    return re.findall(
        r"\d+[.,]?\d*|\b(cero|uno|dos|tres|cuatro|cinco|seis|siete|ocho|nueve|"
        r"diez|once|doce|trece|catorce|quince|veinte|treinta|cuarenta|"
        r"cincuenta|sesenta|setenta|ochenta|noventa|cien|ciento|mil)\b",
        text
    )


# ==============================
# DETECCIÓN DE ERRORES EN NÚMEROS
# ==============================

# Detecta sustituciones en números (ref → hyp)
def number_error_details(results, ground_truth):
    
    errors = Counter()
    
    for item in results:
        audio_id = item["audio_id"]
        
        # Verificamos que exista en el ground truth
        if audio_id in ground_truth:
            
            ref = normalize_text(ground_truth[audio_id])
            hyp = normalize_text(item["text"])
            
            if ref and hyp:
                
                ref_words = ref.split()
                hyp_words = hyp.split()
                
                # Alineamiento entre referencia y predicción
                alignment = process_words(ref, hyp)
                
                # Recorremos los bloques de alineamiento
                for group in alignment.alignments:
                    for chunk in group:
                        
                        # Nos quedamos con sustituciones
                        if chunk.type == "substitute":
                            
                            ref_segment = ref_words[chunk.ref_start_idx:chunk.ref_end_idx]
                            hyp_segment = hyp_words[chunk.hyp_start_idx:chunk.hyp_end_idx]
                            
                            # FILTRO: solo palabra a palabra
                            if len(ref_segment) == 1 and len(hyp_segment) == 1:
                                
                                ref_word = ref_segment[0]
                                hyp_word = hyp_segment[0]
                                
                                # Solo si el original es número
                                if ref_word in extract_numbers(ref):
                                    errors[(ref_word, hyp_word)] += 1
    
    return errors


# ==============================
# CÁLCULO
# ==============================

num_errors_std = number_error_details(results_standardized, ground_truth)
num_errors_proc = number_error_details(results_processed, ground_truth)


# ==============================
# VISUALIZACIÓN
# ==============================

print("Sustituciones en números:")

print("\nStandardized:")
if num_errors_std:
    for (ref, hyp), count in num_errors_std.most_common(10):
        print(f"{ref} → {hyp} ({count})")
else:
    print("sin errores")

print("\nProcessed:")
if num_errors_proc:
    for (ref, hyp), count in num_errors_proc.most_common(10):
        print(f"{ref} → {hyp} ({count})")
else:
    print("sin errores")

Sustituciones en números:

Standardized:
cuatro → cuatroquinientos (1)
cinco → cs95 (1)
seis → yelletarias (1)
uno → un (1)
uno → una (1)
diez → dietas (1)
once → conceptarias (1)

Processed:
seis → yelletarias (1)
uno → un (1)
uno → una (1)
diez → dietas (1)
once → conceptarias (1)


Las sustituciones en números muestran un comportamiento claramente errático, con transformaciones que no mantienen relación semántica con el valor original, como *cinco* → *cs95*, *seis* → *yelletarias* o *once* → *conceptarias*. Este patrón indica que el modelo no reconoce correctamente ciertos números en contexto y los transforma en secuencias fonéticamente similares o directamente irrelevantes.

También se observan casos de simplificación morfológica (*uno* → *un*, *uno* → un*a*), que, aunque menos críticos, pueden afectar a la interpretación si el número tiene valor cuantitativo explícito.

La comparación entre configuraciones muestra una ligera reducción de errores en el *pipeline* procesado, eliminando algunas sustituciones presentes en el *baseline* (*cuatro* → *cuatroquinientos*, *cinco* → *cs95*), lo que sugiere una mejora puntual en la estabilidad del reconocimiento numérico.

Este tipo de error resulta especialmente relevante, ya que los valores numéricos suelen estar asociados a cantidades, superficies o medidas, por lo que una transcripción incorrecta puede alterar completamente el significado de la información extraída.

#### 7.2.3 Resumen cuantitativo del impacto

Con el objetivo de integrar ambos tipos de error, se construye un resumen cuantitativo que recoge el número total de errores en términos de dominio y en valores numéricos para cada configuración.

Este análisis permite evaluar el impacto conjunto de los errores sobre la capacidad del sistema para generar información estructurada.

In [35]:
# Conteo total desde los Counters
domain_std = sum(domain_errors_std.values())
domain_proc = sum(domain_errors_proc.values())

number_std = sum(num_errors_std.values())
number_proc = sum(num_errors_proc.values())


# Construcción de tabla resumen
df_struct_errors = pd.DataFrame({
    "pipeline": ["standardized", "processed"],
    "domain_errors": [domain_std, domain_proc],
    "number_errors": [number_std, number_proc]

})


# Impacto total
df_struct_errors["total_impact"] = (
    df_struct_errors["domain_errors"] +
    df_struct_errors["number_errors"]

)

display(
    df_struct_errors.rename(columns={
        "pipeline": "Pipeline",
        "domain_errors": "Errores en dominio",
        "number_errors": "Errores en números",
        "total_impact": "Impacto total"
    })
)

,Pipeline,Errores en dominio,Errores en números,Impacto total
0,standardized,11,7,18
1,processed,11,5,16


El análisis cuantitativo muestra que el número total de errores con impacto en la estructuración es reducido en ambas configuraciones, con valores de 18 en el *baseline* y 16 en el *pipeline* procesado. Esta diferencia indica una mejora ligera tras el preprocesamiento, aunque su magnitud es limitada en términos absolutos.

En el caso de los errores de dominio, no se observan diferencias entre configuraciones, lo que confirma que el preprocesamiento no influye de forma significativa en la preservación de términos clave. Por el contrario, en los errores numéricos sí se aprecia una pequeña reducción, lo que sugiere una mejora puntual en la transcripción de cantidades.

#### 7.2.4 Conclusiones del análisis de errores con impacto en la estructuración

El análisis de errores con impacto en la estructuración muestra que, aunque su frecuencia es reducida, afectan a elementos críticos de la información, como términos de dominio y valores numéricos.

En el caso de los términos de dominio, los errores se producen principalmente en forma de sustituciones que alteran la forma original de los conceptos (*moniliasis* → *monidiasis*, *cacao* → *cacau*), lo que puede dificultar su identificación en etapas posteriores del *pipeline*. Este comportamiento se mantiene de forma consistente entre configuraciones, sin mejoras relevantes asociadas al preprocesamiento.

En cuanto a los valores numéricos, se observan errores de mayor severidad, con transformaciones que rompen completamente el significado (*cinco* → *cs95*, *seis* → *yelletarias*). Aunque su frecuencia es menor, su impacto es elevado debido a la importancia de las cantidades en la interpretación del contenido.

La comparación entre configuraciones muestra que el preprocesamiento introduce mejoras puntuales en la transcripción de números, pero no modifica de forma significativa los errores asociados a términos de dominio.

### 7.3 Impacto de los errores en etapas posteriores (PLN)

Los errores de transcripción identificados a lo largo del análisis presentan un impacto directo en las etapas posteriores del *pipeline* de PLN, aunque su frecuencia global sea reducida.

Las sustituciones constituyen el principal foco de impacto, especialmente cuando afectan a términos de dominio o a valores numéricos. En estos casos, la alteración de la forma original puede impedir la correcta identificación de entidades o modificar el significado del mensaje, afectando tanto a la extracción de información como a la clasificación.

Por su parte, las inserciones y eliminaciones muestran un comportamiento más estable y, en general, afectan a palabras funcionales, por lo que su impacto semántico es limitado. Sin embargo, en casos puntuales pueden contribuir a distorsionar la estructura del mensaje o dificultar la interpretación contextual.

El análisis específico de términos de dominio confirma que los errores no suelen implicar la desaparición completa de los conceptos, sino su transformación en variantes no reconocibles, lo que representa un problema crítico para tareas de Named Entity Recognition (NER). De forma similar, los errores en números y cantidades, aunque menos frecuentes, presentan un impacto elevado al afectar directamente a valores cuantitativos relevantes.

En conjunto, los resultados evidencian que la calidad del ASR condiciona el rendimiento del sistema de PLN no tanto por la cantidad de errores, sino por su naturaleza y localización dentro del mensaje.

Como línea de mejora, se propone la incorporación de mecanismos de normalización ligera previos al procesamiento lingüístico, basados en diccionarios de dominio y técnicas de similitud léxica, que permitan corregir automáticamente variantes cercanas a términos relevantes sin alterar el contenido original. Este enfoque permitiría aumentar la robustez del sistema frente a errores del ASR sin introducir complejidad innecesaria en el *pipeline*.

## 8. Comparativa de configuraciones de transcripción

Esta sección analiza el impacto de distintas configuraciones del modelo ASR sobre la calidad de las transcripciones, evaluando tanto métricas clásicas (WER, CER) como errores con impacto semántico y eficiencia temporal. El objetivo es identificar la configuración que ofrece el mejor equilibrio entre precisión, robustez y coste computacional.

### 8.1 Variación de parámetros del modelo

#### 8.1.1 Definición de configuraciones

Se definen varias configuraciones del modelo modificando parámetros clave del proceso de decodificación, como el tamaño del beam search y el uso de contexto entre segmentos. Estas configuraciones permiten analizar cómo influyen diferentes estrategias de inferencia en la calidad final de la transcripción.

In [36]:
# Definición de configuraciones a evaluar

configs = {
    "baseline": {
        "language": "es",
        "beam_size": 1,
        "condition_on_previous_text": False,
        "word_timestamps": False,
        "temperature": 0.0
    },
    "beam_3": {
        "language": "es",
        "beam_size": 3,
        "condition_on_previous_text": False,
        "word_timestamps": False,
        "temperature": 0.0
    },
    "with_context": {
        "language": "es",
        "beam_size": 1,
        "condition_on_previous_text": True,
        "word_timestamps": False,
        "temperature": 0.0
    },
    "combined": {
        "language": "es",
        "beam_size": 3,
        "condition_on_previous_text": True,
        "word_timestamps": False,
        "temperature": 0.0
    }
}

#### 8.1.2 Ejecución de transcripciones y medición de tiempo

Cada configuración se evalúa sobre el mismo conjunto de audios, registrando tanto las transcripciones generadas como el tiempo total de ejecución. Esto permite analizar no solo la calidad del resultado, sino también la eficiencia temporal de cada enfoque.

In [ ]:
# Almacenamiento de resultados y tiempos
results_all = {}
execution_times = {}

for config_name, params in configs.items():
    
    results = []
    
    # Inicio del cronómetro
    start = time.time()
    
    for item in tqdm(dataset_processed, desc=f"Transcribiendo: {config_name}"):
        
        segments, _ = model.transcribe(
            str(item["path"]),
            **params
        )
        
        text = " ".join([seg.text for seg in segments]).strip()
        
        results.append({
            "audio_id": item["audio_id"],
            "text": text
        })
    
    # Fin del cronómetro
    end = time.time()
    
    total_time = end - start
    avg_time = total_time / len(dataset_processed)
    
    results_all[config_name] = results
    
    execution_times[config_name] = {
        "total_time": total_time,
        "avg_time_per_audio": avg_time
    }

Transcribiendo: beam_3:  69%|██████▉   | 138/200 [11:49<04:55,  4.76s/it]

### 8.2 Evaluación comparativa

Para cada configuración se realiza una evaluación integrada que combina métricas de error, análisis lingüístico y medición de impacto estructural, permitiendo una comparación completa del comportamiento del sistema.

#### 8.2.1 Caracterización del conjunto de datos

Se calcula la duración total y media del conjunto de audios, lo que permite contextualizar el coste computacional del proceso de transcripción y calcular métricas de eficiencia como el *Real Time Factor* (RTF).

In [ ]:
# Cálculo de duración total y media de los audios

durations = []

for item in dataset_processed:
    y, sr = librosa.load(item["path"], sr=None)
    durations.append(len(y) / sr)

durations = np.array(durations)

total_audio_duration = durations.sum()
avg_audio_duration = durations.mean()

#### 8.2.2 Función de evaluación integrada

Se define una función de evaluación unificada que integra métricas de calidad (WER, CER), análisis de errores lingüísticos y evaluación del impacto sobre elementos críticos como términos de dominio y valores numéricos.

In [ ]:
def evaluate_asr_results(results, ground_truth, label):
    
    output = {}
    
    # =========================
    # MÉTRICAS (WER / CER)
    # =========================
    
    wer_scores = compute_wer(results, ground_truth)
    cer_scores = compute_cer(results, ground_truth)
    
    output["wer_stats"] = compute_robust_stats(wer_scores)
    output["cer_stats"] = compute_robust_stats(cer_scores)
    
    
    # =========================
    # ERRORES LINGÜÍSTICOS
    # =========================
    
    subs = get_substitutions(results, ground_truth)
    ins = get_insertions(results, ground_truth)
    dels = get_deletions(results, ground_truth)
    
    output["errors_summary"] = {
        "substitutions": sum(subs.values()),
        "insertions": sum(ins.values()),
        "deletions": sum(dels.values())
    }
    
    
    # =========================
    # ERRORES DE DOMINIO
    # =========================
    
    domain_errors = domain_error_details(results, ground_truth)
    output["domain_errors"] = sum(domain_errors.values())
    
    
    # =========================
    # ERRORES NUMÉRICOS
    # =========================
    
    number_errors = number_error_details(results, ground_truth)
    output["number_errors"] = sum(number_errors.values())
    
    
    # =========================
    # IMPACTO TOTAL
    # =========================
    
    output["total_impact"] = (
        output["domain_errors"] +
        output["number_errors"]
    )
    
    return output

#### 8.2.3 Evaluación de configuraciones

Se aplica la función de evaluación a cada una de las configuraciones definidas, obteniendo un conjunto homogéneo de resultados que permite su comparación directa.

In [ ]:
# Evaluación de todas las configuraciones definidas en 8.1

evaluations = {}

# Evaluación de todas las configuraciones definidas en 8.1
for config_name, results in results_all.items():
    
    evaluations[config_name] = evaluate_asr_results(
        results,
        ground_truth,
        config_name
    )

#### 8.2.4 Construcción de tabla comparativa

Se construye una tabla comparativa que integra todas las métricas relevantes, incluyendo calidad de transcripción, impacto estructural y eficiencia temporal, facilitando la interpretación conjunta de los resultados.

In [ ]:
# Construimos un DataFrame a partir de los resultados evaluados de cada configuración
# Integra métricas de calidad, impacto estructural y eficiencia temporal
df_comparison = pd.DataFrame([
    {
        "config": name,  # Nombre de la configuración evaluada
        
        # Métricas principales de transcripción
        "wer_mean": eval_data["wer_stats"]["mean"],   # Error medio a nivel de palabra
        "wer_p95": eval_data["wer_stats"]["p95"],     # Percentil 95 (comportamiento en peores casos)
        "cer_mean": eval_data["cer_stats"]["mean"],   # Error medio a nivel de carácter
        
        # Impacto estructural (errores relevantes para el dominio)
        "domain_errors": eval_data["domain_errors"],  # Pérdida de términos clave del dominio
        "number_errors": eval_data["number_errors"],  # Errores en valores numéricos
        "total_impact": eval_data["total_impact"],    # Impacto total combinado
        
        # Tiempo de ejecución
        "execution_time_total": execution_times[name]["total_time"],        # Tiempo total de inferencia
        "execution_time_avg": execution_times[name]["avg_time_per_audio"],  # Tiempo medio por audio
        
        # Eficiencia temporal (RTF: Real Time Factor)
        "rtf": execution_times[name]["total_time"] / total_audio_duration,   # Relación tiempo procesamiento / duración real

        # Tamaño del dataset (contexto clave)
        "n_audios": len(dataset_processed)
    }
    for name, eval_data in evaluations.items()
])

#### 8.2.5 Procesamiento y formateo de resultados

Se aplican transformaciones adicionales para mejorar la legibilidad de los resultados, incluyendo el formateo de tiempos y la normalización de métricas, con el objetivo de facilitar su análisis e interpretación.

In [ ]:
# Convierte segundos a formato legible (minutos/horas)
def format_time(seconds, short=False):
    if short:
        return f"{seconds:.1f}s"
    
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    
    if hours > 0:
        return f"{hours}h {minutes}m {secs}s"
    return f"{minutes}m {secs}s"

# Añadimos columnas formateadas para facilitar la interpretación
df_comparison["execution_time_avg_fmt"] = df_comparison["execution_time_avg"].apply(lambda x: format_time(x, short=True))
df_comparison["execution_time_total_fmt"] = df_comparison["execution_time_total"].apply(format_time)

# Redondeo de RTF
df_comparison["rtf"] = df_comparison["rtf"].round(2)

#### 8.2.6 Análisis comparativo de configuraciones

Se presentan los resultados ordenados según criterios de calidad y eficiencia, permitiendo identificar de forma directa las configuraciones más favorables en función de los distintos indicadores evaluados.

In [ ]:
# Ordenamos por calidad (WER) y eficiencia (RTF) y mostramos columnas clave
display(
    df_comparison.sort_values(["wer_mean", "rtf"])[[
        "config",
        "wer_mean",
        "wer_p95",
        "cer_mean",
        "domain_errors",
        "number_errors",
        "total_impact",
        "rtf",
        "execution_time_avg_fmt", 
        "execution_time_total_fmt"
    ]]
)

# Mostramos información global del conjunto de datos utilizado
print(
    f"Dataset: {len(dataset_processed)} audios | "
    f"Duración media: {avg_audio_duration:.2f}s | "
    f"Total: {format_time(total_audio_duration)}"
)

,config,wer_mean,wer_p95,cer_mean,domain_errors,number_errors,total_impact,rtf,execution_time_avg_fmt,execution_time_total_fmt
0,baseline,0.037864,0.142857,0.013790,11,5,16,0.33,4.5s,14m 56s
1,beam_3,0.039340,0.143006,0.014758,10,6,16,0.36,4.9s,16m 13s
2,with_context,0.040069,0.143527,0.015662,11,6,17,0.32,4.3s,14m 16s
3,combined,0.040604,0.143006,0.016119,10,6,16,0.36,4.9s,16m 19s


Dataset: 200 audios | Duración media: 13.43s | Total: 44m 46s


Los resultados muestran que la configuración *baseline* presenta el mejor comportamiento global, alcanzando los valores más bajos tanto en WER (0.0379) como en CER (0.0138), lo que indica una mayor precisión en la transcripción respecto al resto de configuraciones evaluadas.

El uso de *beam search* con mayor tamaño (*beam_3*) no mejora los resultados, sino que introduce un ligero empeoramiento en todas las métricas, además de incrementar el coste computacional, como se refleja en un mayor RTF (0.36 frente a 0.33) y tiempos de ejecución más elevados.

Por su parte, la configuración *with_context* tampoco aporta mejoras en calidad, mostrando un aumento tanto en WER como en CER. Aunque presenta el mejor rendimiento temporal (RTF = 0.32), esta mejora en eficiencia no compensa la pérdida de precisión observada.

La configuración *combined*, que combina ambos enfoques (*beam search* y uso de contexto), muestra el peor comportamiento global, con los valores más altos de error y un coste computacional elevado, lo que indica que la combinación de estas estrategias no resulta adecuada en este escenario.

En cuanto al impacto estructural, las diferencias entre configuraciones son mínimas. Los errores en términos de dominio se mantienen prácticamente constantes, mientras que los errores numéricos presentan ligeras variaciones sin una mejora clara asociada a ninguna configuración.

Estos resultados evidencian que aumentar la complejidad del proceso de decodificación no implica necesariamente una mejora en la calidad de las transcripciones, y que configuraciones más simples pueden ofrecer un mejor equilibrio entre precisión y eficiencia en este contexto.

#### 8.2.7 Análisis de la distribución del error (WER)

Se analiza la distribución del error a nivel individual mediante visualizaciones, lo que permite identificar la variabilidad entre configuraciones y detectar la presencia de valores atípicos o comportamientos extremos.

In [ ]:
# =========================
# CONSTRUCCIÓN DATAFRAME LONG PARA WER
# =========================

rows = []

for config_name, results in results_all.items():
    
    wer_scores = compute_wer(results, ground_truth)
    
    for score in wer_scores:
        rows.append({
            "config": config_name,
            "wer": score
        })

df_long = pd.DataFrame(rows)


fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Gráfico completo (outliers visibles)
sns.boxplot(
    data=df_long,
    x="config",
    y="wer",
    hue="config",
    palette="Set2",
    legend=False,
    flierprops=dict(marker='o', markerfacecolor='red', markersize=6),
    ax=axes[0]
)

axes[0].set_title("WER (completo)")
axes[0].set_ylabel("WER")
axes[0].grid(axis="y", linestyle="--", alpha=0.3)

# --- Gráfico zoom (zona útil)
sns.boxplot(
    data=df_long,
    x="config",
    y="wer",
    hue="config",
    palette="Set2",
    legend=False,
    showfliers=False,
    ax=axes[1]
)

sns.stripplot(
    data=df_long,
    x="config",
    y="wer",
    color="black",
    size=3,
    alpha=0.4,
    ax=axes[1]
)

axes[1].set_ylim(0, 0.3)
axes[1].set_title("WER (zoom)")
axes[1].grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

NameError: name 'results_all' is not defined

El análisis de la distribución del WER muestra un comportamiento muy similar entre todas las configuraciones evaluadas, en línea con los resultados observados previamente en la tabla comparativa.

En primer lugar, la mediana del error es igual a 0 en todos los casos, lo que indica que más del 50% de las transcripciones no presentan errores. Esto confirma el buen rendimiento general del modelo en condiciones habituales.

Las distribuciones presentan formas prácticamente idénticas, sin diferencias relevantes entre configuraciones. Este comportamiento coincide con las métricas globales, donde las variaciones en WER y CER eran muy reducidas.

En el gráfico completo se aprecian varios valores atípicos, asociados a audios más complejos. Estos casos aparecen en todas las configuraciones, lo que sugiere que los errores se concentran en un subconjunto concreto de audios y no dependen tanto de los parámetros del modelo.

El gráfico con zoom permite ver mejor la zona donde se agrupa la mayoría de los resultados. En este rango, las diferencias entre configuraciones son mínimas, aunque el *baseline* mantiene una ligera ventaja, coherente con lo observado anteriormente.

### 8.3 Selección de configuración óptima

A partir del análisis conjunto de métricas de calidad, impacto estructural y eficiencia temporal, se selecciona la configuración óptima del modelo, justificando la decisión en función de los objetivos del sistema.

A partir del análisis realizado, la configuración *baseline* se selecciona como la opción óptima para el sistema.

Esta decisión se basa en que presenta los mejores valores de WER y CER, manteniendo además un impacto estructural bajo y estable en términos de errores de dominio y numéricos. Aunque las diferencias con el resto de configuraciones son reducidas, ninguna alternativa logra mejorar de forma consistente estos resultados.

Desde el punto de vista computacional, el *baseline* también ofrece un buen equilibrio, con un *Real Time Factor* competitivo y tiempos de ejecución inferiores a las configuraciones que incorporan mayor complejidad, como el aumento del *beam size* o el uso de contexto.

Las configuraciones alternativas no aportan mejoras claras y, en algunos casos, introducen un mayor coste computacional o un ligero empeoramiento en las métricas de calidad.

Por tanto, se opta por una configuración simple y eficiente, evitando añadir complejidad innecesaria al sistema.

## 9. Implementación y ejecución del *pipeline* final de transcripción de audio

Una vez definido el flujo de procesamiento y seleccionada la configuración del modelo más adecuada, se procede a la implementación del *pipeline* final de transcripción.

En esta fase se integran todos los componentes en un flujo completo, utilizando como entrada los audios previamente procesados y aplicando el modelo ASR con la configuración seleccionada. El resultado es un sistema automatizado capaz de generar transcripciones estructuradas en formato JSON de forma consistente y reproducible.

El diseño del *pipeline* mantiene un enfoque modular, lo que facilita la trazabilidad de cada etapa y permite su adaptación en fases posteriores del proyecto, especialmente en su integración con los módulos de procesamiento del lenguaje natural.

### 9.1 Inicialización del entorno y gestión de rutas

En primer lugar, se define un mecanismo flexible para la gestión de rutas del proyecto, permitiendo localizar automáticamente la estructura de directorios y configurar las carpetas de entrada y salida. Este enfoque facilita la portabilidad del *pipeline*, evitando dependencias rígidas de entorno y permitiendo adaptar fácilmente la organización de datos sin modificar la lógica de procesamiento.

In [ ]:
# Define las rutas del proyecto para el pipeline ASR
# Permite seleccionar la carpeta de entrada y fija la salida final del ASR
def configure_paths(
    base_data_dir: str = "data",
    input_folder: str = "processed",
    output_folder: str = "transcriptions"
) -> dict:

    project_root = Path.cwd()

    # Detectar raíz del proyecto
    while not (project_root / base_data_dir).exists():
        project_root = project_root.parent

    data_dir = project_root / base_data_dir

    # Carpeta base de audios
    audio_dir = data_dir / "audio"

    # Carpeta de entrada (processed o standardized)
    input_audio_dir = audio_dir / input_folder

    # Carpeta de transcripciones
    transcriptions_dir = data_dir / output_folder

    # Carpeta final del ASR (clave)
    asr_output_dir = transcriptions_dir / "asr_output"

    # Crear carpetas necesarias
    asr_output_dir.mkdir(parents=True, exist_ok=True)

    return {
        "project_root": project_root,
        "input_audio_dir": input_audio_dir,
        "asr_output_dir": asr_output_dir
    }

### 9.2 Implementación de funciones del *pipeline* de transcripción

A continuación, se implementan las distintas funciones que componen el *pipeline* de transcripción automática. Cada función aborda una etapa específica del flujo, incluyendo la carga de datos, la inicialización del modelo, la generación de transcripciones y el almacenamiento de resultados.

Este diseño modular permite separar claramente las responsabilidades de cada componente, facilitando su validación, mantenimiento y reutilización en distintas fases del sistema.

#### 9.2.1 Carga del *dataset* de audios procesados

En esta fase se realiza la carga de los audios procesados desde el sistema de almacenamiento, construyendo una estructura de datos ligera basada en el identificador y la ruta de cada archivo. Esta representación permite gestionar el conjunto de datos de forma eficiente durante la ejecución del *pipeline* de transcripción.

In [ ]:
# Carga los audios desde el directorio de entrada
# Devuelve una lista de diccionarios con el identificador y la ruta de cada audio
def load_audio_dataset(
    input_audio_dir: Path,
    valid_extensions: set[str] = {".wav"}
) -> list[dict]:

    # Filtrado de archivos de audio válidos
    audio_files = [
        f for f in input_audio_dir.iterdir()
        if f.is_file() and f.suffix.lower() in valid_extensions
    ]

    # Verificación de existencia de audios
    if len(audio_files) == 0:
        raise FileNotFoundError("No se han encontrado audios válidos en el directorio de entrada")

    # Construcción del dataset con identificador y ruta
    dataset = [
        {
            "audio_id": f.stem,
            "path": f
        }
        for f in audio_files
    ]

    return dataset

#### 9.2.2 Inicialización del modelo ASR

Se inicializa el modelo de reconocimiento automático del habla utilizando la configuración previamente seleccionada como óptima. Esta etapa deja el modelo preparado para su uso en la generación de transcripciones sobre el conjunto de audios.

In [ ]:
# Inicializa el modelo ASR con la configuración especificada
# Devuelve el modelo listo para ser utilizado en la transcripción
def load_asr_model(
    model_size: str = "medium",
    device: str = "cpu",
    compute_type: str = "int8"
):

    model = WhisperModel(
        model_size,
        device=device,
        compute_type=compute_type
    )

    return model

#### 9.2.3 Transcripción de audios

En esta etapa se aplica el modelo ASR sobre el conjunto de audios, generando una transcripción completa para cada archivo. El proceso se realiza de forma iterativa, reconstruyendo el texto final a partir de los segmentos producidos por el modelo.

In [ ]:
# Aplica el modelo ASR para transcribir los audios del dataset
# Permite configurar parámetros de inferencia, con valores por defecto
def transcribe_audio_dataset(
    audio_dataset: list[dict],
    model,
    asr_params: dict = None
) -> list[dict]:

    # Definición de parámetros por defecto si no se proporcionan externamente
    if asr_params is None:
        asr_params = {
            "language": "es",                    # Fuerza el idioma español
            "beam_size": 1,                      # Decodificación greedy (rápida)
            "condition_on_previous_text": False, # Evita dependencia entre segmentos
            "word_timestamps": False,            # No se necesitan timestamps por palabra
            "temperature": 0.0                   # Salida determinista
        }

    results = []  # Lista donde se almacenarán las transcripciones

    # Iteración sobre cada audio con barra de progreso
    for item in tqdm(audio_dataset, desc="Transcribiendo audios"):

        # Ejecución del modelo ASR sobre el audio
        segments, _ = model.transcribe(
            str(item["path"]),  # Ruta del archivo de audio
            **asr_params        # Parámetros de inferencia
        )

        # Reconstrucción del texto completo a partir de los segmentos generados
        text = " ".join([seg.text for seg in segments]).strip()

        # Almacenamiento del resultado estructurado
        results.append({
            "audio_id": item["audio_id"],  # Identificador único del audio
            "text": text                   # Transcripción completa
        })

    return results  # Devuelve todas las transcripciones generadas

#### 9.2.4 Generación de salida estructurada

Las transcripciones generadas se almacenan en formato JSON, manteniendo una estructura uniforme que incluye el identificador del audio y el texto transcrito. Este formato facilita su integración en etapas posteriores del *pipeline*, como la clasificación o la extracción de entidades.

In [ ]:
# Guarda cada transcripción como un JSON independiente
def save_transcriptions_json(
    transcriptions: list[dict],
    output_dir: Path
) -> None:

    for item in transcriptions:
        output_path = output_dir / f"{item['audio_id']}.json"

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(item, f, ensure_ascii=False, indent=4)

### 9.3 Ejecución del *pipeline* de transcripción de audio

Una vez definidas todas las funciones, se ejecuta el *pipeline* completo de forma secuencial. Este bloque coordina la aplicación ordenada de cada etapa sobre el conjunto de audios, desde su carga inicial hasta la generación de las transcripciones finales en formato estructurado.

La estructura lineal del flujo facilita la comprensión del proceso global y permite seguir de forma clara la transformación de los datos desde el audio de entrada hasta su representación textual.

<div style="border-left: 5px solid #ff4d4d; padding: 10px; background-color: #2b2b2b; color: #f0f0f0;">
<strong style="color: red">⚠️ Advertencia:</strong> En esta implementación se asume que el orden de los elementos en el <i>dataset</i> se mantiene constante a lo largo del <i>pipeline</i>. Esta simplificación es válida en este contexto experimental, pero puede provocar desalineaciones entre audios y transcripciones si se modifica el <i>dataset</i> o se ejecutan etapas de forma independiente. En entornos productivos se recomienda utilizar identificadores únicos para garantizar la trazabilidad.
</div>

In [ ]:
print("Inicializando pipeline ASR...")

# Configuración de rutas
paths = configure_paths()
print("Rutas configuradas correctamente")

# Carga de audios
print("Cargando audios...")
audio_dataset = load_audio_dataset(paths["input_audio_dir"])
print(f"Audios encontrados: {len(audio_dataset)}")

# Inicialización del modelo
print("Inicializando modelo ASR...")
model_size = "medium"
device = "cpu"
compute_type = "int8"

model = load_asr_model(
    model_size=model_size,
    device=device,
    compute_type=compute_type
)
print("Modelo ASR cargado correctamente")

# Transcripción
print("Transcribiendo audios...")
results = transcribe_audio_dataset(audio_dataset, model)
print(f"Audios transcritos: {len(results)}")

# Ruta de salida (ya viene definida en configure_paths)
asr_output_dir = paths["asr_output_dir"]
print(f"Ruta de salida: {asr_output_dir}")

# Guardado
print("Guardando transcripciones...")
save_transcriptions_json(results, asr_output_dir)
print("Transcripciones guardadas correctamente")

print("Pipeline completado correctamente")

Inicializando pipeline ASR...
Rutas configuradas correctamente
Cargando audios...
Audios encontrados: 200
Inicializando modelo ASR...
Modelo ASR cargado correctamente
Transcribiendo audios...


Transcribiendo audios: 100%|██████████| 200/200 [15:03<00:00,  4.52s/it]

Audios transcritos: 200
Ruta de salida: /Volumes/EXTENSION/GitHub/TFM/data/transcriptions/asr_output
Guardando transcripciones...
Transcripciones guardadas correctamente
Pipeline completado correctamente


### 9.4 Validación del *pipeline* de transcripción de audio

En este apartado se realiza una verificación sistemática del comportamiento del *pipeline* una vez generadas las transcripciones. El objetivo es asegurar la consistencia de los resultados, comprobando que el número de salidas coincide con el número de audios de entrada y que cada registro contiene la estructura esperada.

Adicionalmente, se valida la unicidad de los identificadores y se detectan posibles transcripciones vacías, lo que permite identificar errores en la inferencia o problemas asociados a la calidad del audio.

Este proceso garantiza la integridad del conjunto de datos generado y su idoneidad para su uso en etapas posteriores del *pipeline*.

In [ ]:
# Verificación básica del pipeline de transcripción

print("\nIniciando validación...")

# Conteo
assert len(results) == len(audio_dataset), "Error: número de transcripciones no coincide"
print("✔ Conteo correcto")

# Estructura + contenido + unicidad
audio_ids = set()
empty_texts = []

for item in results:
    assert "audio_id" in item, "Error: falta 'audio_id'"
    assert "text" in item, "Error: falta 'text'"
    
    assert isinstance(item["text"], str), "Error: 'text' no es string"
    
    # Detección de textos vacíos (sin romper pipeline)
    if item["text"].strip() == "":
        empty_texts.append(item["audio_id"])
    
    # Unicidad
    assert item["audio_id"] not in audio_ids, f"audio_id duplicado: {item['audio_id']}"
    audio_ids.add(item["audio_id"])

print("✔ Estructura y contenido correctos")
print("✔ Unicidad de audio_id verificada")

# Reporte de textos vacíos
if empty_texts:
    print(f"⚠ Textos vacíos detectados: {len(empty_texts)}")
    print("Ejemplo:", empty_texts[:5])
else:
    print("✔ No se detectaron textos vacíos")

# Dataset limpio para etapas posteriores (opcional)
results_clean = [item for item in results if item["text"].strip() != ""]

print(f"✔ Registros válidos para NLP: {len(results_clean)}")

# Muestra para inspección
print("\n=== Muestra de transcripciones ===")
sample = pd.DataFrame(results).sample(n=min(5, len(results)), random_state=42)
display(sample)

print("✔ Validación completada correctamente")


Iniciando validación...
✔ Conteo correcto
✔ Estructura y contenido correctos
✔ Unicidad de audio_id verificada
✔ No se detectaron textos vacíos
✔ Registros válidos para NLP: 200

=== Muestra de transcripciones ===


,audio_id,text
95,AUDIO_00096,Hoy se hizo aplicación foliar con biofertiliza...
15,AUDIO_00016,"Estamos viendo bastante roya en las hojas, sob..."
30,AUDIO_00031,En estos momentos estamos en plena cosecha de ...
158,AUDIO_00159,Hoy se aplicó tratamiento orgánico para contro...
128,AUDIO_00129,Entonces organizando las actividades del día e...


✔ Validación completada correctamente


La validación del *pipeline* confirma que el proceso de transcripción se ha ejecutado correctamente sobre el conjunto completo de audios. El número de registros generados coincide con el número de entradas, la estructura de salida es consistente y no se detectan transcripciones vacías, lo que indica un funcionamiento estable del modelo ASR en este conjunto de datos.

Adicionalmente, la unicidad de los identificadores garantiza la trazabilidad de cada audio dentro del *pipeline*, permitiendo su uso fiable en etapas posteriores de procesamiento del lenguaje natural.

## 10. Conclusiones

El desarrollo de este *notebook* ha permitido construir y validar un *pipeline* completo de transcripción automática de audio, partiendo de audios previamente procesados y evaluando distintas configuraciones del modelo Whisper.

Los resultados obtenidos muestran que el modelo base ofrece el mejor equilibrio entre calidad y eficiencia, manteniendo niveles de error bajos tanto en WER como en CER, sin penalizar el tiempo de ejecución. Las configuraciones más complejas no aportan mejoras significativas y, en algunos casos, introducen una ligera degradación del rendimiento.

El análisis de errores ha permitido caracterizar el comportamiento real del sistema. La mayoría de las transcripciones se mantienen estables entre configuraciones, y las diferencias observadas se concentran en un subconjunto reducido de audios. Los errores más frecuentes son de tipo general (sustituciones, inserciones y eliminaciones), mientras que los errores que afectan a términos de dominio o a expresiones numéricas tienen una incidencia muy baja en el conjunto total analizado.

Desde la perspectiva del procesamiento del lenguaje natural, la calidad de las transcripciones obtenidas permite abordar con fiabilidad tareas de clasificación y extracción de entidades en el *notebook* 3. Aunque se identifican errores puntuales que afectan a términos específicos, su baja frecuencia limita su impacto global. En la fase de normalización posterior, estos casos podrán ser tratados mediante reglas o mecanismos de corrección específicos cuando sea necesario.

El *pipeline* funciona de forma consistente y permite generar transcripciones que se pueden usar directamente en el resto del sistema. A partir de aquí, las mejoras no están tanto en modificar el modelo ASR, sino en adaptarlo mejor al dominio y en tratar de forma específica los pocos errores que realmente tienen impacto.